<a href="https://colab.research.google.com/github/comet-toolkit/comet_training/blob/main/CoMet_HYPERNETS_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Training Session - CoMet Toolkit: Uncertainties made easy**

#Exercise 5: Dealing with uncertainties for time series

## Objectives

In this exercise you will apply the CoMet Toolkit to handle uncertainties when processing measurements separated in time one at a time:

* Combine uncertainties based on temporal error correlation
* Store uncertainty components for each processed file
* Combine the processed files at a later stage

### **Introduction**
Practically, when collecting and distributing measurement data, it is often required to process these data as they come in. Typically one or more (e.g. one day of measurements) measurements are processed from raw data, to the data product with the measurand of interest. These individual data products are then often combined at a later stage. 

In order to add uncertainties for such a use case, it is important that the error correlations are properly handled when multiple data products are combined. In order to do this, one needs to know the error correlation between the different data products (i.e. temporal error correlation). As we have seen in the previous exercise, such information can be captured in the metadata of the datafiles. It is however not always trivial to store such information, as it does not make sense to store an actual temporal error correlation matrix in each file, as we do not know a priori how many measurements there would be.

We will again use the example of the spectrometer measurements of the cress, but this time process the repeat measurements individually (imagining these would e.g. be measurements from subsequent days).

## *Step 1* - Set up the environment

First, we again install punpy.

In [ ]:
!pip install punpy>=1.0.6


Please hit `Runtime > Restart Session` to properly load these packages into your Google Colab environment...

We also import all the relevant python packages used in this training:

In [ ]:
import xarray as xr
import numpy as np
from punpy import MCPropagation
import matplotlib.pyplot as plt

## *Step 2* - Reading in satellite data

In [ ]:
# Load datasets from NetCDF files
spectralon_ds = xr.load_dataset("comet_training/example_data/Spectralon.nc")
cress_ds = xr.load_dataset("comet_training/example_data/Cress.nc")

# Load Spectralon reference reflectance
spectralon_ref_ds = xr.load_dataset("comet_training/example_data/spectralon_reflectance.nc")

# Extract wavelengths and digital number (DN) data
wavelength = spectralon_ds['wavelength'].values

# Normalise DN measurements by integration time to make them comparable across different measurement settings
# This converts raw DN counts to a per-unit-time basis
dn_panel_repeats = spectralon_ds['digital_number'].values / spectralon_ds.attrs['integration_time_ms']      # shape: (n_wavelength, n_panel_repeats)
dn_cress_repeats = cress_ds['digital_number'].values / cress_ds.attrs['integration_time_ms']             # shape: (n_wavelength, n_cress_repeats)

def calculate_stats(dn_repeats):
    """
    Calculate mean and standard deviation across repeated measurements.

    :param dn_repeats: repeated measurements array (shape: n_wavelengths × n_repeats)
    :returns: tuple of (mean, standard_deviation) arrays
    """
    mean = np.mean(dn_repeats, axis=1)
    std = np.std(dn_repeats, axis=1, ddof=1)
    return mean, std

dn_panel_mean, dn_panel_std = calculate_stats(dn_panel_repeats)
dn_cress_mean, dn_cress_std = calculate_stats(dn_cress_repeats)

In [ ]:
# Load and interpolate Spectralon reflectance to measured wavelength grid
from scipy import interpolate

wavelength_ref = spectralon_ref_ds['wavelength'].values
reflectance_ref = spectralon_ref_ds['reflectance'].values

# Create linear interpolation function
f_interp = interpolate.interp1d(wavelength_ref, reflectance_ref, kind='linear', 
                                bounds_error=False, fill_value='extrapolate')

# Interpolate to measured wavelength grid
rho_panel = f_interp(wavelength)
# Create wavelength-dependent uncertainty for Spectralon reflectance (in %)
# Based on calibration certificate data
u_rho_panel_percent = np.zeros_like(wavelength)
u_rho_panel_percent[(wavelength >= 465) & (wavelength <= 780)] = 0.55
u_rho_panel_percent[(wavelength > 780) & (wavelength <= 1000)] = 1.60

# Convert uncertainty from percentage to absolute value
u_rho_panel = (u_rho_panel_percent / 100.0) * rho_panel

In [ ]:
def reflectance_model(dn_surface, dn_panel, rho_panel):
    """Calibrate surface reflectance from DN using reference panel."""
    return (dn_surface / dn_panel) * rho_panel

In [ ]:
def save_results(i,rho_cress,u_random_rho_cress,u_syst_rho_cress):
    """Save results to a NetCDF file."""
    ds = xr.Dataset(
        {
            'rho_cress': (('wavelength'), rho_cress),
            'u_random_rho_cress': (('wavelength'), u_random_rho_cress),
            'u_syst_rho_cress': (('wavelength'), u_syst_rho_cress)
        },
        coords={'wavelength': wavelength}
    )
    ds.to_netcdf(f'calibrated_reflectance_{i}.nc')

def load_results(i):
    """Load results from a NetCDF file."""
    ds = xr.load_dataset(f'calibrated_reflectance_{i}.nc')
    return ds['rho_cress'].values, ds['u_random_rho_cress'].values, ds['u_syst_rho_cress'].values

From these we can determine the reflectance in each band from the mean over the Region of Interest (ROI), and its uncertainty (combination of noise and spatial variability) from the standard deviation between pixels:

In [ ]:
# Initialise punpy's Monte Carlo propagation object with 10,000 iterations
prop = MCPropagation(10000)

for i in range(len(dn_cress_repeats)):
    rho_cress = reflectance_model(dn_cress_repeats[i], dn_panel_mean, rho_panel)
    u_random_rho_cress = prop.propagate_random(
    reflectance_model,
        [dn_cress_repeats[i], dn_panel_mean, rho_panel],
        [dn_cress_std, dn_panel_std, None]
    )
    u_syst_rho_cress = prop.propagate_systematic(
        reflectance_model,
        [dn_cress_repeats[i], dn_panel_mean, rho_panel],
        [None, None, u_rho_panel]
    )
    save_results(i,rho_cress,u_random_rho_cress,u_syst_rho_cress)


In [ ]:
rho_cress = np.empty((len(wavelength), 10))
u_random_rho_cress = np.zeros_like(rho_cress)
u_syst_rho_cress = np.zeros_like(rho_cress)

for i in range(len(dn_cress_repeats)):
    rho_cress[:,i] = load_results(i)[0]
    u_random_rho_cress[:,i] = load_results(i)[1]
    u_syst_rho_cress[:,i] = load_results(i)[2]

def calculate_mean(data):
    return np.mean(data, axis=1)

L1_avg_ur = prop.propagate_random(calculate_mean,[rho_cress],
      [u_random_rho_cress])
print("L1 average random uncertainty:", L1_avg_ur)
L1_avg_us = prop.propagate_systematic(calculate_mean,[rho_cress],
      [u_syst_rho_cress])
print("L1 average systematic uncertainty:", L1_avg_us)
L1_avg_ut_errcorr = (L1_avg_ur**2 + L1_avg_us**2)**0.5
print("L1 average total uncertainty (errcorr):", L1_avg_ut_errcorr)